In [0]:
CREATE OR REPLACE STREAMING TABLE ORDERS_BRONZE_RAW
COMMENT "LOADING ORDERS DATA FROM VOLUME to bronze layer"
AS 
SELECT * ,
_METADATA.file_name AS FILE_NAME,
CURRENT_TIMESTAMP AS INGEST_TIME
FROM CLOUD_FILEs('/Volumes/dlt/dlt_practice/dlt_data/orders/','CSV',MAP("cloudfiles.inferColumnTypes","True"))

In [0]:
CREATE OR REPLACE STREAMING TABLE orders_silver_cleaned(
  constraint valid_order expect (order_id is not null)ON VIOLATION drop row,
  constraint valid_customerid expect (customer_id is not null)ON VIOLATION drop row)
as select orderid as order_id,
customerid as customer_id,
orderdate as order_date,
totalamount as total_amount,
status,
file_name,
ingest_time
from stream(live.ORDERS_BRONZE_RAW);


In [0]:
create or replace streaming table orders_silver;
apply changes into orders_silver
from stream(live.orders_silver_cleaned)
keys(order_id)
sequence by ingest_time;


In [0]:
create or replace materialized view city_wise_sales
select c.city,sum(o.total_amount) as total_sales
from live.orders_silver o join live.customer_silver c
on o.customer_id=c.customer_id
group by city;
